# Titanic Survival Prediction — Preprocessing, Model Comparison & Clustering

**Problem:** given details about a passenger (class, sex, age, fare, …), predict whether they survived the Titanic disaster.

**Steps in this notebook**
1. Load and inspect the raw data
2. Pre-process: drop unhelpful columns, fill missing values, encode categories
3. Baseline model: Logistic Regression
4. Decision Tree — readable rules and an overfitting experiment
5. Random Forest
6. Compare all models (test set + 5-fold cross-validation)
7. Final model: Random Forest — evaluation, feature importance, save
8. Bonus: K-Means clustering (unsupervised)

> The dataset is **not** included in this repo. See the README for how to download `train.csv` from Kaggle into the `data/` folder.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works both locally (data/train.csv) and on Kaggle
LOCAL_PATH = "../data/train.csv"
KAGGLE_PATH = "/kaggle/input/competitions/titanic/train.csv"

DATA_PATH = LOCAL_PATH if os.path.exists(LOCAL_PATH) else KAGGLE_PATH
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
df.head()

## 1. Inspect the data

In [ ]:
print("Data shape (rows, columns):", df.shape)
print("\nData types:\n", df.dtypes)

In [ ]:
# Missing values in each column
df.isnull().sum()

Three columns have missing values:
- **Age** — 177 missing → fill with the median
- **Cabin** — 687 of 891 missing (~77%) → drop the column
- **Embarked** — 2 missing → fill with the most common port (mode)

## 2. Pre-processing

### 2.1 Drop unnecessary columns
`PassengerId`, `Name` and `Ticket` are identifiers / free text with no direct signal, and `Cabin` is mostly empty.

In [ ]:
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
print("Columns after dropping:", df.columns.tolist())

### 2.2 Fill missing values

In [ ]:
# Age -> median (robust to outliers)
median_age = df['Age'].median()
print("Median age:", median_age)
df.fillna({'Age': median_age}, inplace=True)

# Embarked -> mode (most frequent port)
mode_embarked = df['Embarked'].mode()[0]
print("Mode of Embarked:", mode_embarked)
df.fillna({'Embarked': mode_embarked}, inplace=True)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

### 2.3 Convert categorical columns to numbers
The models only understand numbers, so `Sex` and `Embarked` are mapped to integers.

In [ ]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df.head()

### 2.4 Train / test split
Kaggle's `test.csv` has no `Survived` labels, so 20% of `train.csv` is held out as a test set to measure accuracy.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Survived', axis=1)
Y = df['Survived']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

## 3. Baseline: Logistic Regression
A simple, interpretable model to compare everything else against.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train, Y_train)
log_pred = log_model.predict(X_test)
print("Logistic Regression accuracy:", accuracy_score(Y_test, log_pred))

# Positive coefficient = raises survival chance, negative = lowers it
pd.Series(log_model.coef_[0], index=X.columns).sort_values()

Being female (`Sex = 1`) is the strongest push towards survival, while a higher `Pclass` number (2nd/3rd class) lowers it.

## 4. Decision Tree
A tree you can read as IF/ELSE rules.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, Y_train)
tree_pred = tree_model.predict(X_test)
print("Decision Tree accuracy:", accuracy_score(Y_test, tree_pred))

In [ ]:
# The tree as IF/ELSE rules
print(export_text(tree_model, feature_names=list(X.columns)))

### 4.1 Overfitting experiment
What happens if the tree is allowed to ask more and more questions (greater depth)?

In [ ]:
depths = [1, 2, 3, 5, 8, 12, 20, 50]
train_scores, test_scores = [], []
for depth in depths:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42)
    t.fit(X_train, Y_train)
    train_scores.append(accuracy_score(Y_train, t.predict(X_train)))
    test_scores.append(accuracy_score(Y_test, t.predict(X_test)))
    print(f"Depth {depth:>2} | Train accuracy {train_scores[-1]:.3f} | Test accuracy {test_scores[-1]:.3f}")

plt.plot(range(len(depths)), train_scores, marker='o', label='Train')
plt.plot(range(len(depths)), test_scores, marker='o', label='Test')
plt.xticks(range(len(depths)), depths)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: train vs test accuracy")
plt.legend()
plt.show()

Training accuracy keeps climbing (to ~0.98) while test accuracy stays around 0.80. The deep tree is **memorising** the training data rather than learning patterns that generalise — that gap is overfitting.

## 5. Random Forest
Many decision trees, each trained on a random sample of rows and columns, vote on the answer. Averaging many trees reduces the overfitting a single deep tree suffers from.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
forest_model.fit(X_train, Y_train)
forest_pred = forest_model.predict(X_test)
print("Random Forest accuracy:", accuracy_score(Y_test, forest_pred))

## 6. Compare the models
The test set has only 179 passengers, so one or two predictions can move accuracy noticeably. **5-fold cross-validation** trains and tests each model 5 times on different splits for a more reliable comparison.

In [ ]:
from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": (log_model, log_pred),
    "Decision Tree (depth=3)": (tree_model, tree_pred),
    "Random Forest": (forest_model, forest_pred),
}

rows = []
for name, (m, pred) in models.items():
    cv = cross_val_score(m, X, Y, cv=5)
    rows.append({"Model": name,
                 "Test accuracy": round(accuracy_score(Y_test, pred), 3),
                 "CV mean": round(cv.mean(), 3),
                 "CV std": round(cv.std(), 3)})

results = pd.DataFrame(rows).set_index("Model")
results

Random Forest scores highest, so it is used as the **final model**. Logistic Regression stays as the baseline it is measured against.

## 7. Final model: Random Forest

In [ ]:
final_model = forest_model

# Rows = actual (0 died, 1 survived), columns = predicted
cm = confusion_matrix(Y_test, forest_pred)
pd.DataFrame(cm, index=['Actual: died', 'Actual: survived'],
             columns=['Predicted: died', 'Predicted: survived'])

In [ ]:
# Which columns did the forest find most useful?
importance = pd.Series(final_model.feature_importances_, index=X.columns).sort_values()
importance.plot(kind='barh', title='Random Forest feature importance')
plt.xlabel("Importance")
plt.show()
importance.sort_values(ascending=False)

**Why does Fare rank above Pclass here, when Logistic Regression ranked Pclass higher?**
Tree-based importance favours columns with many distinct values (Fare and Age can be split at hundreds of points; Pclass only at 2). Fare also carries much of the same information as Pclass, so the forest credits Fare for it. Both models agree that **Sex** is one of the strongest signals.

In [ ]:
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(final_model, "../models/titanic_random_forest.joblib")
print("Model saved")

## 8. Bonus: K-Means clustering (unsupervised)
**Goal:** group passengers by Age and Fare **without** telling the model who survived, then check whether the groups still differ in survival.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Work on a copy so the cluster column never leaks into the model features
df_cluster = df.copy()

# Scale first: Fare ranges 0-512 while Age ranges 0-80.
# Without scaling, K-Means would group almost entirely by Fare.
cluster_data = StandardScaler().fit_transform(df_cluster[['Age', 'Fare']])

### 8.1 Elbow plot — choosing the number of groups

In [ ]:
inertia_list = []
for k in range(1, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(cluster_data)
    inertia_list.append(km.inertia_)

plt.plot(range(1, 7), inertia_list, marker='o')
plt.xlabel("K (number of groups)")
plt.ylabel("Spread (inertia)")
plt.title("Elbow Plot")
plt.show()

The curve bends around **K = 3**, so 3 groups are used.

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df_cluster['Group'] = kmeans.fit_predict(cluster_data)

print("Passengers in each group:")
print(df_cluster['Group'].value_counts())

print("\nAverage Age, Fare and Pclass of each group:")
print(df_cluster.groupby('Group')[['Age', 'Fare', 'Pclass']].mean().round(1))

In [ ]:
# K-Means never saw 'Survived'. Do the groups still differ in survival?
print("Share of passengers who survived per group:")
df_cluster.groupby('Group')['Survived'].mean().round(2)

Even without the labels, the groups found from Age and Fare alone have clearly different survival rates — wealthier (higher-fare) passengers survived more often.

## Conclusion

- Cleaned the raw data: dropped 4 columns, filled missing Age (median) and Embarked (mode), encoded Sex and Embarked.
- Compared three models; **Random Forest** performed best and is saved as the final model, with Logistic Regression as the baseline.
- The depth experiment showed how a single decision tree overfits as it grows.
- K-Means found passenger groups with different survival rates without ever seeing the labels.

**Ideas to improve**
- Feature engineering: `FamilySize = SibSp + Parch + 1`, title from `Name` (Mr/Mrs/Miss/Master)
- One-hot encode `Embarked` (it has no natural order)
- Tune Random Forest (`max_depth`, `n_estimators`, `min_samples_leaf`) with `GridSearchCV`
- Apply the same preprocessing to `test.csv` and submit predictions to Kaggle